<div dir="rtl" style="text-align: right; line-height: 1.9;">

# تمرین کامپیوتری ۴ یادگیری عمیق
## تشخیص پنج نوع گل با شبکه عصبی پیچشی (CNN)


در این نوت‌بوک یک شبکه‌ی عصبی پیچشی برای تشخیص پنج کلاس گل طراحی، تنظیم و ارزیابی می‌شود. مراحل شامل آماده‌سازی داده، نرمال‌سازی، طراحی مدل، کنترل overfitting، مقایسه‌ی optimizer و نرخ یادگیری، آموزش نهایی، محاسبه‌ی Accuracy/Precision/Recall، رسم نمودار خطا و پیش‌بینی دو تصویر است.

</div>

<div dir="rtl" style="text-align: right; line-height: 1.9;">

## 1. کتابخانه‌ها و تنظیمات تکرارپذیری

برای قابل‌مقایسه بودن آزمایش‌ها، seed ثابت استفاده شده است. ابعاد تصویر و اندازه‌ی batch با توجه به اندازه‌ی دیتاست و هزینه‌ی محاسباتی انتخاب شده‌اند.

</div>

In [ ]:
from pathlib import Path
import tarfile
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

SEED = 123
IMG_HEIGHT = 180
IMG_WIDTH = 180
IMAGE_SIZE = (IMG_HEIGHT, IMG_WIDTH)
BATCH_SIZE = 32
EXPERIMENT_EPOCHS = 5
FINAL_EPOCHS = 30

tf.keras.utils.set_random_seed(SEED)
print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

<div dir="rtl" style="text-align: right; line-height: 1.9;">

## 2. بارگذاری دیتاست

کد ابتدا فایل یا پوشه‌ی موجود در کنار نوت‌بوک را استفاده می‌کند. اگر فایل محلی موجود نباشد، دیتاست رسمی TensorFlow دانلود می‌شود؛ بنابراین همین نوت‌بوک در سیستم محلی و Google Colab قابل اجرا است.

</div>

In [ ]:
project_dir = Path.cwd()
local_dataset_dir = project_dir / "flower_photos"
local_archive = project_dir / "flower_photos.tgz"

def safe_extract(tar, destination):
    """Extract an archive only when every member remains inside destination."""
    destination = destination.resolve()
    for member in tar.getmembers():
        member_path = (destination / member.name).resolve()
        if member_path != destination and destination not in member_path.parents:
            raise ValueError(f"Unsafe archive member: {member.name}")
    tar.extractall(destination)

if local_dataset_dir.exists():
    flowers_root = local_dataset_dir
    print("Using extracted local dataset:", flowers_root)
elif local_archive.exists():
    print("Extracting local archive:", local_archive)
    with tarfile.open(local_archive, "r:gz") as archive:
        safe_extract(archive, project_dir)
    flowers_root = local_dataset_dir
else:
    print("Local dataset was not found; downloading the TensorFlow copy.")
    downloaded_path = tf.keras.utils.get_file(
        "flower_photos",
        "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz",
        untar=True,
    )
    flowers_root = Path(downloaded_path)

assert flowers_root.exists(), f"Dataset directory does not exist: {flowers_root}"
print("Dataset path:", flowers_root.resolve())

In [ ]:
class_directories = sorted(path for path in flowers_root.iterdir() if path.is_dir())
class_counts = {path.name: len(list(path.glob("*.jpg"))) for path in class_directories}
image_count = sum(class_counts.values())

print("Total number of images:", image_count)
for class_name, count in class_counts.items():
    print(f"{class_name:12s}: {count}")

plt.figure(figsize=(8, 4))
plt.bar(class_counts.keys(), class_counts.values(), color="teal")
plt.title("Number of images in each class")
plt.ylabel("Image count")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

<div dir="rtl" style="text-align: right; line-height: 1.9;">

## 3. جداسازی داده‌های آموزش و ارزیابی

۸۰٪ تصاویر برای آموزش و ۲۰٪ برای ارزیابی در نظر گرفته شده است. استفاده از seed یکسان تضمین می‌کند که دو مجموعه هم‌پوشانی نداشته باشند. داده‌ی ارزیابی در آموزش مدل استفاده نمی‌شود.

</div>

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    flowers_root,
    validation_split=0.20,
    subset="training",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    flowers_root,
    validation_split=0.20,
    subset="validation",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)
print("Number of classes:", num_classes)

In [ ]:
plt.figure(figsize=(10, 8))
for images, labels in train_ds.take(1):
    for index in range(9):
        ax = plt.subplot(3, 3, index + 1)
        plt.imshow(images[index].numpy().astype("uint8"))
        plt.title(class_names[int(labels[index])])
        plt.axis("off")
plt.tight_layout()
plt.show()

<div dir="rtl" style="text-align: right; line-height: 1.9;">

## 4. نرمال‌سازی و بهینه‌سازی pipeline

مقادیر خام پیکسل در بازه‌ی ۰ تا ۲۵۵ هستند. لایه‌ی Rescaling(1./255) در ابتدای مدل آن‌ها را به بازه‌ی ۰ تا ۱ تبدیل می‌کند. قرار دادن نرمال‌سازی داخل مدل باعث می‌شود همین تبدیل هنگام پیش‌بینی تصاویر جدید نیز فراموش نشود. همچنین prefetch زمان انتظار CPU/GPU را کاهش می‌دهد.

</div>

In [ ]:
for sample_images, _ in train_ds.take(1):
    print("Raw pixel range:", float(tf.reduce_min(sample_images)), "to", float(tf.reduce_max(sample_images)))
    normalized_sample = layers.Rescaling(1.0 / 255)(sample_images)
    print(
        "Normalized pixel range:",
        float(tf.reduce_min(normalized_sample)),
        "to",
        float(tf.reduce_max(normalized_sample)),
    )

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

<div dir="rtl" style="text-align: right; line-height: 1.9;">

## 5. انتخاب معماری، هایپرپارامترها و جلوگیری از Overfitting

معماری پایه شامل چهار بلوک کانولوشن است. تعداد فیلترها به‌تدریج از ۳۲ به ۲۵۶ افزایش می‌یابد تا ویژگی‌های ساده و سپس ویژگی‌های پیچیده‌تر استخراج شوند. اندازه‌ی kernel برابر ۳×۳ و تابع فعال‌سازی ReLU است.

روش‌های به‌کاررفته برای کاهش overfitting:

- **Data augmentation:** چرخش، جابه‌جایی، zoom، contrast و flip تصادفی فقط هنگام آموزش.
- **Batch Normalization:** پایدارتر کردن آموزش و کاهش حساسیت به مقداردهی اولیه.
- **Global Average Pooling:** کاهش شدید تعداد پارامترها نسبت به Flatten.
- **Dropout:** غیرفعال کردن تصادفی بخشی از نورون‌ها هنگام آموزش.
- **L2 regularization:** جریمه کردن وزن‌های بسیار بزرگ.
- **EarlyStopping و ReduceLROnPlateau:** توقف در زمان مناسب و کاهش نرخ یادگیری هنگام توقف پیشرفت.

مقادیر اولیه‌ی batch size، dropout و L2 بر اساس اندازه‌ی کوچک دیتاست انتخاب شده‌اند. optimizer و learning rate با آزمایش کنترل‌شده در بخش بعد انتخاب می‌شوند.

</div>

In [ ]:
def build_cnn(dropout_rate=0.40, l2_strength=1e-4):
    data_augmentation = keras.Sequential(
        [
            layers.RandomFlip("horizontal", seed=SEED),
            layers.RandomRotation(0.10, seed=SEED),
            layers.RandomZoom(0.10, seed=SEED),
            layers.RandomTranslation(0.08, 0.08, seed=SEED),
            layers.RandomContrast(0.10, seed=SEED),
        ],
        name="data_augmentation",
    )

    model = keras.Sequential(
        [
            layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
            data_augmentation,
            layers.Rescaling(1.0 / 255, name="normalization"),

            layers.Conv2D(32, 3, padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.MaxPooling2D(),

            layers.Conv2D(64, 3, padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.MaxPooling2D(),

            layers.Conv2D(128, 3, padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.MaxPooling2D(),

            layers.Conv2D(256, 3, padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.MaxPooling2D(),

            layers.GlobalAveragePooling2D(),
            layers.Dropout(dropout_rate),
            layers.Dense(
                128,
                activation="relu",
                kernel_regularizer=regularizers.l2(l2_strength),
            ),
            layers.Dropout(0.30),
            layers.Dense(num_classes, activation="softmax"),
        ],
        name="flower_cnn",
    )
    return model

model = build_cnn()
model.summary()

<div dir="rtl" style="text-align: right; line-height: 1.9;">

## 6. مقایسه‌ی Optimizer و Learning Rate

برای مقایسه‌ی منصفانه، معماری، داده‌ها، seed و تعداد epoch در تمام آزمایش‌ها ثابت هستند و فقط optimizer و learning rate تغییر می‌کنند. معیار انتخاب، بیشترین دقت روی داده‌ی ارزیابی است؛ استفاده از دقت آموزش به‌تنهایی می‌تواند یک مدل overfit را به‌اشتباه بهترین مدل نشان دهد.

</div>

In [ ]:
def make_optimizer(name, learning_rate):
    name = name.lower()
    if name == "adam":
        return keras.optimizers.Adam(learning_rate=learning_rate)
    if name == "rmsprop":
        return keras.optimizers.RMSprop(learning_rate=learning_rate)
    if name == "sgd":
        return keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9, nesterov=True)
    raise ValueError(f"Unknown optimizer: {name}")

experiment_configs = [
    ("Adam", 1e-3),
    ("Adam", 3e-4),
    ("RMSprop", 1e-3),
    ("RMSprop", 3e-4),
    ("SGD", 1e-2),
    ("SGD", 1e-3),
]

print(f"{len(experiment_configs)} controlled experiments will be run.")

In [ ]:
experiment_results = []

for optimizer_name, learning_rate in experiment_configs:
    print("\n" + "=" * 72)
    print(f"Experiment: optimizer={optimizer_name}, learning_rate={learning_rate:g}")

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    experiment_model = build_cnn()
    experiment_model.compile(
        optimizer=make_optimizer(optimizer_name, learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    experiment_history = experiment_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EXPERIMENT_EPOCHS,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=2,
                restore_best_weights=True,
            )
        ],
        verbose=2,
    )

    best_epoch_index = int(np.argmax(experiment_history.history["val_accuracy"]))
    experiment_results.append(
        {
            "optimizer": optimizer_name,
            "learning_rate": learning_rate,
            "epochs_run": len(experiment_history.history["loss"]),
            "best_epoch": best_epoch_index + 1,
            "train_accuracy_at_best": experiment_history.history["accuracy"][best_epoch_index],
            "best_val_accuracy": experiment_history.history["val_accuracy"][best_epoch_index],
            "best_val_loss": experiment_history.history["val_loss"][best_epoch_index],
        }
    )

    del experiment_model

print("\nOptimizer comparison completed.")

In [ ]:
sorted_experiments = sorted(
    experiment_results,
    key=lambda row: row["best_val_accuracy"],
    reverse=True,
)

header = (
    f"{'Optimizer':10s} {'LR':>10s} {'Epoch':>7s} "
    f"{'Train Acc':>12s} {'Val Acc':>12s} {'Val Loss':>12s}"
)
print(header)
print("-" * len(header))
for row in sorted_experiments:
    print(
        f"{row['optimizer']:10s} {row['learning_rate']:10.1e} "
        f"{row['best_epoch']:7d} {row['train_accuracy_at_best']:12.4f} "
        f"{row['best_val_accuracy']:12.4f} {row['best_val_loss']:12.4f}"
    )

best_experiment = sorted_experiments[0]
BEST_OPTIMIZER = best_experiment["optimizer"]
BEST_LEARNING_RATE = best_experiment["learning_rate"]

labels = [f"{row['optimizer']}\nLR={row['learning_rate']:.0e}" for row in experiment_results]
scores = [row["best_val_accuracy"] for row in experiment_results]
plt.figure(figsize=(11, 5))
bars = plt.bar(labels, scores, color="slateblue")
plt.ylim(max(0, min(scores) - 0.08), min(1.0, max(scores) + 0.05))
plt.ylabel("Best validation accuracy")
plt.title("Optimizer and learning-rate comparison")
plt.grid(axis="y", alpha=0.25)
for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width() / 2, score + 0.005, f"{score:.3f}", ha="center")
plt.tight_layout()
plt.show()

print(f"Selected optimizer: {BEST_OPTIMIZER}")
print(f"Selected learning rate: {BEST_LEARNING_RATE:g}")

<div dir="rtl" style="text-align: right; line-height: 1.9;">

## 7. آموزش مدل نهایی

مدل نهایی با بهترین optimizer و نرخ یادگیری حاصل از آزمایش‌های بالا آموزش داده می‌شود. EarlyStopping بهترین وزن‌ها را بر اساس validation loss بازیابی می‌کند و ReduceLROnPlateau در صورت متوقف شدن پیشرفت، نرخ یادگیری را کاهش می‌دهد.

</div>

In [ ]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

model = build_cnn()
model.compile(
    optimizer=make_optimizer(BEST_OPTIMIZER, BEST_LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=2,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        "best_flower_cnn.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINAL_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

<div dir="rtl" style="text-align: right; line-height: 1.9;">

## 8. نمودار دقت و خطا طی فرایند آموزش

فاصله‌ی زیاد بین منحنی آموزش و ارزیابی نشانه‌ی overfitting است. هدف روش‌های منظم‌سازی بالا، کوچک نگه داشتن این فاصله و کاهش validation loss است.

</div>

In [ ]:
epochs_range = range(1, len(history.history["loss"]) + 1)

plt.figure(figsize=(13, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history.history["accuracy"], label="Training accuracy")
plt.plot(epochs_range, history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy during training")
plt.legend()
plt.grid(alpha=0.25)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history.history["loss"], label="Training loss")
plt.plot(epochs_range, history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss during training")
plt.legend()
plt.grid(alpha=0.25)

plt.tight_layout()
plt.show()

<div dir="rtl" style="text-align: right; line-height: 1.9;">

## 9. ارزیابی Accuracy، Precision و Recall

از آنجا که مسئله چندکلاسه و تعداد نمونه‌های کلاس‌ها یکسان نیست، علاوه بر Accuracy، مقدار Precision و Recall هر کلاس و میانگین macro گزارش می‌شود. میانگین macro به هر کلاس وزن یکسان می‌دهد و برای مشاهده‌ی ضعف مدل روی کلاس‌های کم‌نمونه مناسب است.

</div>

In [ ]:
def collect_predictions(trained_model, dataset):
    all_true = []
    all_pred = []
    for batch_images, batch_labels in dataset:
        probabilities = trained_model.predict(batch_images, verbose=0)
        all_true.append(batch_labels.numpy())
        all_pred.append(np.argmax(probabilities, axis=1))
    return np.concatenate(all_true), np.concatenate(all_pred)

def evaluate_and_report(trained_model, dataset, split_name):
    loss, accuracy = trained_model.evaluate(dataset, verbose=0)
    y_true, y_pred = collect_predictions(trained_model, dataset)
    cm = tf.math.confusion_matrix(y_true, y_pred, num_classes=num_classes).numpy()

    true_positives = np.diag(cm).astype(float)
    predicted_positives = cm.sum(axis=0).astype(float)
    actual_positives = cm.sum(axis=1).astype(float)
    precision = np.divide(
        true_positives,
        predicted_positives,
        out=np.zeros_like(true_positives),
        where=predicted_positives != 0,
    )
    recall = np.divide(
        true_positives,
        actual_positives,
        out=np.zeros_like(true_positives),
        where=actual_positives != 0,
    )

    print(f"\n{split_name} results")
    print("=" * 55)
    print(f"Loss: {loss:.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro precision: {precision.mean():.4f}")
    print(f"Macro recall: {recall.mean():.4f}")
    print(f"\n{'Class':12s} {'Precision':>12s} {'Recall':>12s} {'Support':>10s}")
    print("-" * 50)
    for index, class_name in enumerate(class_names):
        print(
            f"{class_name:12s} {precision[index]:12.4f} "
            f"{recall[index]:12.4f} {int(actual_positives[index]):10d}"
        )

    return {
        "split": split_name,
        "loss": loss,
        "accuracy": accuracy,
        "macro_precision": precision.mean(),
        "macro_recall": recall.mean(),
        "precision_per_class": precision,
        "recall_per_class": recall,
        "confusion_matrix": cm,
    }

In [ ]:
train_metrics = evaluate_and_report(model, train_ds, "Training set")
val_metrics = evaluate_and_report(model, val_ds, "Validation set")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metrics in zip(axes, [train_metrics, val_metrics]):
    matrix = metrics["confusion_matrix"]
    image = ax.imshow(matrix, cmap="Blues")
    ax.set_title(f"{metrics['split']} confusion matrix")
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_xticks(range(num_classes), class_names, rotation=35, ha="right")
    ax.set_yticks(range(num_classes), class_names)
    for row in range(num_classes):
        for col in range(num_classes):
            ax.text(col, row, matrix[row, col], ha="center", va="center")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

<div dir="rtl" style="text-align: right; line-height: 1.9;">

## 10. پیش‌بینی دو تصویر دلخواه

دو تصویر از مجموعه‌ی ارزیابی انتخاب می‌شوند؛ بنابراین مدل آن‌ها را هنگام آموزش ندیده است. عنوان هر تصویر، برچسب واقعی، پیش‌بینی مدل و میزان اطمینان را نشان می‌دهد.

</div>

In [ ]:
validation_images, validation_labels = next(iter(val_ds))
selected_images = validation_images[:2]
selected_labels = validation_labels[:2].numpy()
prediction_probabilities = model.predict(selected_images, verbose=0)
predicted_labels = np.argmax(prediction_probabilities, axis=1)

plt.figure(figsize=(10, 4))
for index in range(2):
    ax = plt.subplot(1, 2, index + 1)
    plt.imshow(selected_images[index].numpy().astype("uint8"))
    true_name = class_names[int(selected_labels[index])]
    predicted_name = class_names[int(predicted_labels[index])]
    confidence = 100 * float(np.max(prediction_probabilities[index]))
    color = "green" if true_name == predicted_name else "red"
    plt.title(
        f"True: {true_name}\nPredicted: {predicted_name} ({confidence:.1f}%)",
        color=color,
    )
    plt.axis("off")
plt.tight_layout()
plt.show()

<div dir="rtl" style="text-align: right; line-height: 1.9;">

## 11. پاسخ نهایی به سؤال‌های تمرین

### سؤال ۱: بالاترین دقت مدل چقدر است؟

بالاترین دقت معتبر، بیشترین دقت روی داده‌ی ارزیابی است، نه دقت آموزش. سلول زیر مقدار دقیق آن، epoch متناظر، دقت آموزش و معیارهای Precision/Recall را مستقیماً از اجرای مدل گزارش می‌کند.

### سؤال ۲: بهترین optimizer و نرخ یادگیری کدام است؟

بهترین ترکیب بر اساس بیشترین validation accuracy در آزمایش‌های کنترل‌شده انتخاب می‌شود. جدول و نمودار بخش ۶ نتایج تمام ترکیب‌های Adam، RMSprop و SGD را نشان می‌دهند و سلول زیر پاسخ نهایی را چاپ می‌کند.

</div>

In [ ]:
best_final_epoch_index = int(np.argmax(history.history["val_accuracy"]))
best_final_val_accuracy = history.history["val_accuracy"][best_final_epoch_index]
train_accuracy_at_best_epoch = history.history["accuracy"][best_final_epoch_index]
generalization_gap = train_metrics["accuracy"] - val_metrics["accuracy"]

print("FINAL ANSWERS")
print("=" * 72)
print(
    f"1) Highest validation accuracy of the final model: "
    f"{best_final_val_accuracy:.4f} ({100 * best_final_val_accuracy:.2f}%)"
)
print(f"   It was observed at epoch {best_final_epoch_index + 1}.")
print(f"   Training accuracy at that epoch: {train_accuracy_at_best_epoch:.4f}")
print(f"   Final training accuracy after restoring best weights: {train_metrics['accuracy']:.4f}")
print(f"   Final validation accuracy: {val_metrics['accuracy']:.4f}")
print(f"   Validation macro precision: {val_metrics['macro_precision']:.4f}")
print(f"   Validation macro recall: {val_metrics['macro_recall']:.4f}")
print(f"   Train-validation accuracy gap: {generalization_gap:.4f}")
print()
print(
    f"2) Best optimizer in the controlled comparison: {BEST_OPTIMIZER}, "
    f"learning rate={BEST_LEARNING_RATE:g}"
)
print(
    f"   Its best validation accuracy during comparison: "
    f"{best_experiment['best_val_accuracy']:.4f}"
)

<div dir="rtl" style="text-align: right; line-height: 1.9;">

## 12. تحلیل نتایج اجرای نهایی

- **بالاترین دقت ارزیابی مدل نهایی ۷۳٫۹۸٪** و مربوط به epoch شماره‌ی ۳۰ است. دقت مدل پس از بازیابی بهترین وزن‌ها روی داده‌ی آموزش ۷۴٫۷۶٪ و روی داده‌ی ارزیابی ۷۳٫۹۸٪ است.
- اختلاف دقت آموزش و ارزیابی فقط **۰٫۷۸ واحد درصد** است. این اختلاف کم، همراه با روند نزولی validation loss، نشان می‌دهد روش‌های Data Augmentation، Dropout، L2، Batch Normalization و Global Average Pooling توانسته‌اند overfitting را به‌خوبی کنترل کنند.
- مقدار **Macro Precision برابر ۷۳٫۷۰٪** و **Macro Recall برابر ۷۳٫۱۱٪** روی داده‌ی ارزیابی به دست آمد. نزدیک بودن این دو مقدار به Accuracy نشان می‌دهد عملکرد مدل فقط ناشی از کلاس‌های پرتعداد نیست.
- بهترین ترکیب در مقایسه‌ی کنترل‌شده‌ی پنج‌epoch، **SGD با momentum و نرخ یادگیری ۰٫۰۱** بود که دقت ارزیابی ۶۴٫۰۳٪ به دست آورد. پس از آموزش کامل همین ترکیب و کاهش خودکار نرخ یادگیری، دقت به ۷۳٫۹۸٪ رسید.
- در مقایسه‌ی optimizerها، RMSprop با نرخ ۰٫۰۰۱ به دقت ۶۱٫۸۵٪ رسید و دوم شد. ترکیب‌های دارای نرخ یادگیری کوچک‌تر در پنج epoch فرصت کافی برای همگرایی نداشتند؛ به همین دلیل انتخاب فقط با دقت آموزش انجام نشد و معیار validation accuracy مبنا قرار گرفت.
- مدل در کلاس **sunflowers** بهترین Recall را با مقدار ۹۰٫۷۹٪ دارد. ضعیف‌ترین Recall مربوط به کلاس **roses** با مقدار ۵۶٫۶۷٪ است؛ confusion matrix نشان می‌دهد بخش قابل‌توجهی از تصاویر roses با tulips اشتباه شده‌اند.
- هر دو تصویر انتخاب‌شده از داده‌ی ارزیابی به‌درستی تشخیص داده شدند: تصویر dandelion با اطمینان ۷۷٫۷٪ و تصویر daisy با اطمینان ۴۶٫۸٪.

### جمع‌بندی پاسخ سؤال‌ها

1. بالاترین دقت معتبر مدل روی داده‌ی ارزیابی **۷۳٫۹۸٪** است.
2. بهترین optimizer در آزمایش انجام‌شده **SGD با momentum=0.9 و learning rate=0.01** است.

</div>

In [ ]:
model.save("flower_cnn_final.keras")
print("Final model saved as flower_cnn_final.keras")
print("Best checkpoint saved as best_flower_cnn.keras")